In [1]:
import torch
import torch.nn as nn
import torchvision.models as models

# Let's set up the device (uses GPU if you have one, otherwise CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Create dummy images to test our models. 
# Format: (Batch Size, Channels, Height, Width)
# Most modern networks (AlexNet onward) expect 224x224 RGB (3 channels) images.
modern_dummy_image = torch.randn(1, 3, 224, 224).to(device)

# LeNet, being older, traditionally expects 32x32 Grayscale (1 channel) images.
lenet_dummy_image = torch.randn(1, 1, 32, 32).to(device)

Using device: cpu


In [2]:
# PyTorch doesn't have LeNet built-in because it's so simple, so we write it ourselves!
class LeNet(nn.Module):
    def __init__(self):
        super(LeNet, self).__init__()
        self.conv1 = nn.Conv2d(1, 6, kernel_size=5)
        self.pool1 = nn.AvgPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5)
        self.pool2 = nn.AvgPool2d(kernel_size=2, stride=2)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10) # 10 outputs (digits 0-9)

    def forward(self, x):
        x = torch.tanh(self.conv1(x))
        x = self.pool1(x)
        x = torch.tanh(self.conv2(x))
        x = self.pool2(x)
        x = x.view(-1, 16 * 5 * 5) # Flatten the image for the linear layers
        x = torch.tanh(self.fc1(x))
        x = torch.tanh(self.fc2(x))
        x = self.fc3(x)
        return x

lenet_model = LeNet().to(device)
output = lenet_model(lenet_dummy_image)
print(f"LeNet Output Shape: {output.shape} (1 image, 10 class probabilities)")

LeNet Output Shape: torch.Size([1, 10]) (1 image, 10 class probabilities)


In [3]:
# For AlexNet and beyond, PyTorch has them pre-built!
# We set weights='DEFAULT' if we want the pre-trained version, or None for a blank model.
alexnet = models.alexnet(weights=None).to(device)

output = alexnet(modern_dummy_image)
print(f"AlexNet Output Shape: {output.shape} (1 image, 1000 ImageNet classes)")

AlexNet Output Shape: torch.Size([1, 1000]) (1 image, 1000 ImageNet classes)


In [4]:
# We'll use VGG16 (which has 16 weight layers)
vgg16 = models.vgg16(weights=None).to(device)

output = vgg16(modern_dummy_image)
print(f"VGG16 Output Shape: {output.shape}")

VGG16 Output Shape: torch.Size([1, 1000])


In [5]:
# We'll use ResNet-18 (a lighter, 18-layer version of ResNet)
resnet18 = models.resnet18(weights=None).to(device)

output = resnet18(modern_dummy_image)
print(f"ResNet18 Output Shape: {output.shape}")

ResNet18 Output Shape: torch.Size([1, 1000])


In [6]:
# We'll use DenseNet-121
densenet = models.densenet121(weights=None).to(device)

output = densenet(modern_dummy_image)
print(f"DenseNet121 Output Shape: {output.shape}")

DenseNet121 Output Shape: torch.Size([1, 1000])


In [7]:
def count_parameters(model):
    """Returns the total number of trainable weights in a model."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# Create a dictionary of our models
models_dict = {
    "LeNet-5": lenet_model,
    "AlexNet": alexnet,
    "VGG-16": vgg16,
    "ResNet-18": resnet18,
    "DenseNet-121": densenet
}

print("-" * 50)
print(f"{'Architecture':<15} | {'Total Parameters':<20}")
print("-" * 50)

for name, model in models_dict.items():
    param_count = count_parameters(model)
    # Format the number with commas for human readability
    print(f"{name:<15} | {param_count:,}")
    
print("-" * 50)

--------------------------------------------------
Architecture    | Total Parameters    
--------------------------------------------------
LeNet-5         | 61,706
AlexNet         | 61,100,840
VGG-16          | 138,357,544
ResNet-18       | 11,689,512
DenseNet-121    | 7,978,856
--------------------------------------------------


In [8]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

# 1. Setup device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Define image transformations (Resize to 32x32 and Normalize)
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)) # Standard MNIST mean and std dev
])

# 3. Download and load the MNIST dataset
train_dataset = torchvision.datasets.MNIST(root='./data', train=True, transform=transform, download=True)
test_dataset = torchvision.datasets.MNIST(root='./data', train=False, transform=transform, download=True)

# 4. Create DataLoaders for batching
train_loader = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=64, shuffle=False)

# Grab a single batch to test our pipeline later
images, labels = next(iter(train_loader))
images, labels = images.to(device), labels.to(device)
print(f"Loaded a batch of images with shape: {images.shape}") # Expect: [64, 1, 32, 32]

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 9.91M/9.91M [00:06<00:00, 1.53MB/s]


Extracting ./data\MNIST\raw\train-images-idx3-ubyte.gz to ./data\MNIST\raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 28.9k/28.9k [00:00<00:00, 94.4kB/s]


Extracting ./data\MNIST\raw\train-labels-idx1-ubyte.gz to ./data\MNIST\raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 1.65M/1.65M [00:01<00:00, 900kB/s] 


Extracting ./data\MNIST\raw\t10k-images-idx3-ubyte.gz to ./data\MNIST\raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 4.54k/4.54k [00:00<00:00, 2.53MB/s]

Extracting ./data\MNIST\raw\t10k-labels-idx1-ubyte.gz to ./data\MNIST\raw

Loaded a batch of images with shape: torch.Size([64, 1, 32, 32])


In [12]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# FIX: Changed resize from 32x32 to 224x224 so modern networks don't collapse the spatial dimensions
transform = transforms.Compose([
    transforms.Resize((224, 224)), 
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)) 
])

train_dataset = torchvision.datasets.MNIST(root='./data', train=True, transform=transform, download=True)
test_dataset = torchvision.datasets.MNIST(root='./data', train=False, transform=transform, download=True)

train_loader = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=64, shuffle=False)

images, labels = next(iter(train_loader))
images, labels = images.to(device), labels.to(device)
print(f"Loaded a batch of images with shape: {images.shape}")

Loaded a batch of images with shape: torch.Size([64, 1, 224, 224])


In [13]:
import torchvision.models as models

# --- 1. LeNet-5 FIX ---
# Added an AdaptiveAvgPool2d at the end of the features so it can handle 224x224 smoothly
class LeNet5(nn.Module):
    def __init__(self):
        super(LeNet5, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 6, kernel_size=5), nn.Tanh(),
            nn.AvgPool2d(kernel_size=2, stride=2),
            nn.Conv2d(6, 16, kernel_size=5), nn.Tanh(),
            nn.AvgPool2d(kernel_size=2, stride=2),
            nn.AdaptiveAvgPool2d((5, 5)) # <--- This guarantees the output is always 5x5 for the linear layer!
        )
        self.classifier = nn.Sequential(
            nn.Linear(16 * 5 * 5, 120), nn.Tanh(),
            nn.Linear(120, 84), nn.Tanh(),
            nn.Linear(84, 10)
        )
    def forward(self, x):
        return self.classifier(self.features(x).view(-1, 16 * 5 * 5))

lenet = LeNet5().to(device)

# --- 2. AlexNet Tweak ---
alexnet = models.alexnet(weights=None)
alexnet.features[0] = nn.Conv2d(1, 64, kernel_size=11, stride=4, padding=2) 
alexnet.classifier[6] = nn.Linear(4096, 10) 
alexnet = alexnet.to(device)

# --- 3. VGG-16 Tweak ---
vgg16 = models.vgg16(weights=None)
vgg16.features[0] = nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1) 
vgg16.classifier[6] = nn.Linear(4096, 10) 
vgg16 = vgg16.to(device)

# --- 4. ResNet-18 Tweak ---
resnet18 = models.resnet18(weights=None)
resnet18.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False) 
resnet18.fc = nn.Linear(512, 10) 
resnet18 = resnet18.to(device)

# --- 5. DenseNet-121 Tweak ---
densenet = models.densenet121(weights=None)
densenet.features.conv0 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
densenet.classifier = nn.Linear(1024, 10) 
densenet = densenet.to(device)

print("All models successfully modified for 224x224 input and transferred to device!")

All models successfully modified for 224x224 input and transferred to device!


In [14]:
models_dict = {
    "LeNet-5": lenet,
    "AlexNet": alexnet,
    "VGG-16": vgg16,
    "ResNet-18": resnet18,
    "DenseNet-121": densenet
}

print("Verifying structural outputs on real MNIST batches:\n")
for name, model in models_dict.items():
    model.eval() # Set to evaluation mode
    with torch.no_grad():
        output = model(images)
    print(f"✓ {name:<12} successfully processed input shape {list(images.shape)} -> Output shape: {list(output.shape)}")

Verifying structural outputs on real MNIST batches:

✓ LeNet-5      successfully processed input shape [64, 1, 224, 224] -> Output shape: [64, 10]
✓ AlexNet      successfully processed input shape [64, 1, 224, 224] -> Output shape: [64, 10]
✓ VGG-16       successfully processed input shape [64, 1, 224, 224] -> Output shape: [64, 10]
✓ ResNet-18    successfully processed input shape [64, 1, 224, 224] -> Output shape: [64, 10]
✓ DenseNet-121 successfully processed input shape [64, 1, 224, 224] -> Output shape: [64, 10]


In [15]:
def train_and_evaluate(model_name, model, train_loader, test_loader, epochs=1):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    
    print(f"\n--- Training {model_name} ---")
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for batch_idx, (data, targets) in enumerate(train_loader):
            data, targets = data.to(device), targets.to(device)
            
            # Forward pass
            outputs = model(data)
            loss = criterion(outputs, targets)
            
            # Backward pass & optimize
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            if batch_idx % 300 == 0:
                print(f"Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}")
        
        # Quick Evaluation
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for data, targets in test_loader:
                data, targets = data.to(device), targets.to(device)
                outputs = model(data)
                _, predicted = torch.max(outputs.data, 1)
                total += targets.size(0)
                correct += (predicted == targets).sum().item()
        
        accuracy = 100 * correct / total
        print(f"Epoch [{epoch+1}/{epochs}] Complete | Avg Loss: {running_loss/len(train_loader):.4f} | Test Accuracy: {accuracy:.2f}%")

# Example usage (Uncomment the line below to train LeNet on real data instantly!):
# train_and_evaluate("LeNet-5", lenet, train_loader, test_loader, epochs=1)

In [16]:
print("Starting the comparative training process...")
print("Note: Depending on your GPU, this might take a while!\n")

# Loop through all 5 models and train them for 1 epoch each
for name, model in models_dict.items():
    # We call the function you just defined!
    train_and_evaluate(
        model_name=name, 
        model=model, 
        train_loader=train_loader, 
        test_loader=test_loader, 
        epochs=1  # Keep it at 1 for a quick benchmark test
    )

print("\nAll models have been trained and evaluated!")

Starting the comparative training process...
Note: Depending on your GPU, this might take a while!


--- Training LeNet-5 ---
Batch 0/938 | Loss: 2.3338
Batch 300/938 | Loss: 0.3911
Batch 600/938 | Loss: 0.2627
Batch 900/938 | Loss: 0.1511
Epoch [1/1] Complete | Avg Loss: 0.4217 | Test Accuracy: 94.02%

--- Training AlexNet ---
Batch 0/938 | Loss: 2.3032
Batch 300/938 | Loss: 0.1560
Batch 600/938 | Loss: 0.0274
Batch 900/938 | Loss: 0.0377
Epoch [1/1] Complete | Avg Loss: 0.2953 | Test Accuracy: 97.93%

--- Training VGG-16 ---
Batch 0/938 | Loss: 2.3383
Batch 300/938 | Loss: 0.2013
Batch 600/938 | Loss: 0.0928
Batch 900/938 | Loss: 0.3196
Epoch [1/1] Complete | Avg Loss: 0.3948 | Test Accuracy: 98.58%

--- Training ResNet-18 ---
Batch 0/938 | Loss: 2.3984
Batch 300/938 | Loss: 0.0191
Batch 600/938 | Loss: 0.0339
Batch 900/938 | Loss: 0.0209
Epoch [1/1] Complete | Avg Loss: 0.1014 | Test Accuracy: 97.95%

--- Training DenseNet-121 ---
Batch 0/938 | Loss: 2.3469
Batch 300/938 | Loss: 0.0